In [ ]:
"""01_data_exploration.ipynb — Preliminary exploration of ClinVar missense variants.

Goal: load the ClinVar variant_summary, filter down to the labeled missense
SNPs this project trains on, and sanity-check the class balance and data quality
before any feature extraction.
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the project's own modules (features.py, download_clinvar.py) importable.
# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
CLINVAR_FILE = DATA_DIR / "variant_summary.txt.gz"

# Display options so wide bioinformatics tables are actually readable.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(f"Project root: {PROJECT_ROOT}")
print(f"ClinVar file exists: {CLINVAR_FILE.exists()}")

In [ ]:
# The file is tab-delimited and gzipped. compression="infer" reads .gz directly.
# low_memory=False avoids pandas' mixed-dtype warnings on this wide file.
df = pd.read_csv(
    CLINVAR_FILE,
    sep="\t",
    compression="infer",
    low_memory=False,
    dtype=str,            # read everything as string first; coerce deliberately later
)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# What columns do we have, and how complete is each?
print("Columns:\n", list(df.columns), "\n")

# Completeness overview — fraction of non-null values per column.
completeness = df.notna().mean().sort_values()
completeness.to_frame("non_null_fraction")

In [ ]:
# 1. One genome assembly only — ClinVar lists the same variant under both
#    GRCh37 and GRCh38, so not filtering would double-count.
df_38 = df[df["Assembly"] == "GRCh38"].copy()

# 2. Single nucleotide variants only (a missense change is a single base swap).
df_snv = df_38[df_38["Type"] == "single nucleotide variant"].copy()

print(f"GRCh38 rows:        {len(df_38):,}")
print(f"  of which SNVs:    {len(df_snv):,}")

In [ ]:
# See the full landscape of label values before deciding what to keep.
df_snv["ClinicalSignificance"].value_counts().head(20)

In [ ]:
# Standard ClinVar label-cleaning: keep only confident pathogenic / benign calls.
# "Likely" calls are conventionally folded in; uncertain/conflicting are dropped.
PATHOGENIC = {"Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"}
BENIGN     = {"Benign", "Likely benign", "Benign/Likely benign"}

def to_label(sig: str):
    if sig in PATHOGENIC:
        return 1
    if sig in BENIGN:
        return 0
    return np.nan   # uncertain, conflicting, drug-response, etc. → excluded

df_snv["label"] = df_snv["ClinicalSignificance"].map(to_label)

# Keep only rows that resolved to a clean binary label.
df_labeled = df_snv.dropna(subset=["label"]).copy()
df_labeled["label"] = df_labeled["label"].astype(int)

print(f"Confidently labeled SNVs: {len(df_labeled):,}")

In [ ]:
counts = df_labeled["label"].value_counts().rename({0: "benign", 1: "pathogenic"})
print(counts)
print(f"\nPathogenic fraction: {df_labeled['label'].mean():.1%}")

In [ ]:
# The HGVS Name looks like:
#   NM_000546.6(TP53):c.524G>A (p.Arg175His)
# Your feature extractor needs the p.(...) protein-level part.
has_protein = df_labeled["Name"].str.contains(r"p\.", regex=True, na=False)
print(f"Labeled SNVs with a protein-level (p.) consequence: {has_protein.sum():,} "
      f"({has_protein.mean():.1%})")

# Peek at a few real Names to confirm the format matches what features.py expects.
df_labeled.loc[has_protein, ["GeneSymbol", "Name", "ClinicalSignificance", "label"]].head()

In [ ]:
df_labeled.loc[has_protein, "GeneSymbol"].value_counts().head(15)